In [11]:
!pip install -q transformers datasets evaluate accelerate tensorboard sentencepiece

In [14]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split

In [74]:
print("PyTorch Version :", torch.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device :", device)

if device == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

PyTorch Version : 2.10.0+cu128
Device : cuda
GPU : Tesla T4


In [75]:
base_path = "/kaggle/input/datasets/pariza/bbc-news-summary/BBC News Summary"

article_folder = os.path.join(base_path, "News Articles")
summary_folder = os.path.join(base_path, "Summaries")

print("Articles Folder :", article_folder)
print("Summaries Folder :", summary_folder)

Articles Folder : /kaggle/input/datasets/pariza/bbc-news-summary/BBC News Summary/News Articles
Summaries Folder : /kaggle/input/datasets/pariza/bbc-news-summary/BBC News Summary/Summaries


In [76]:
texts = []
summaries = []
categories = []

category_list = [
    "business",
    "entertainment",
    "politics",
    "sport",
    "tech"
]

for category in category_list:

    article_path = os.path.join(article_folder, category)
    summary_path = os.path.join(summary_folder, category)

    files = sorted(os.listdir(article_path))

    for file in files:

        article_file = os.path.join(article_path, file)
        summary_file = os.path.join(summary_path, file)

        with open(article_file, "r", encoding="latin-1") as f:
            article = f.read()

        with open(summary_file, "r", encoding="latin-1") as f:
            summary = f.read()

        texts.append(article)
        summaries.append(summary)
        categories.append(category)

In [77]:
df = pd.DataFrame({
    "Text": texts,
    "Summary": summaries,
    "Category": categories
})

print("Dataset Created Successfully!")

Dataset Created Successfully!


In [78]:
# Take only 500 samples for quick fine-tuning

sample_df = df.sample(
    n=500,
    random_state=42
).reset_index(drop=True)

print("Sample Dataset Shape :", sample_df.shape)

Sample Dataset Shape : (500, 3)


In [79]:
sample_df = sample_df[["Text", "Summary", "Category"]]

sample_df.head()

,Text,Summary,Category
0,UK house prices dip in November\n\nUK house pr...,All areas saw a rise in annual house price inf...,business
1,LSE 'sets date for takeover deal'\n\nThe Londo...,A Â£1.3bn offer from Deutsche Boerse has alrea...,business
2,Harinordoquy suffers France axe\n\nNumber eigh...,Harinordoquy was a second-half replacement in ...,sport
3,Barclays shares up on merger talk\n\nShares in...,Shares in UK banking group Barclays have risen...,business
4,Campaign 'cold calls' questioned\n\nLabour and...,Assistant information commissioner Phil Jones ...,politics


In [25]:
print(sample_df["Category"].value_counts())

Category
business         129
sport            115
tech              92
politics          84
entertainment     80
Name: count, dtype: int64


In [80]:
train_df, temp_df = train_test_split(
    sample_df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("Training Samples :", len(train_df))
print("Temporary Samples :", len(temp_df))

Training Samples : 400
Temporary Samples : 100


In [81]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Validation Samples :", len(val_df))
print("Test Samples :", len(test_df))

Validation Samples : 50
Test Samples : 50


In [28]:
print("="*40)

print("Training Set :", train_df.shape)

print("Validation Set :", val_df.shape)

print("Test Set :", test_df.shape)

print("="*40)

Training Set : (400, 3)
Validation Set : (50, 3)
Test Set : (50, 3)


In [82]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [83]:
train_dataset = Dataset.from_pandas(train_df)

val_dataset = Dataset.from_pandas(val_df)

test_dataset = Dataset.from_pandas(test_df)

In [84]:
print(train_dataset)

print(val_dataset)

print(test_dataset)

Dataset({
    features: ['Text', 'Summary', 'Category'],
    num_rows: 400
})
Dataset({
    features: ['Text', 'Summary', 'Category'],
    num_rows: 50
})
Dataset({
    features: ['Text', 'Summary', 'Category'],
    num_rows: 50
})


In [33]:
print(train_dataset[0])

{'Text': 'Economy \'stronger than forecast\'\n\nThe UK economy probably grew at a faster rate in the third quarter than the 0.4% reported, according to Bank of England deputy governor Rachel Lomax.\n\nPrivate sector business surveys suggest a stronger economy than official estimates, Ms Lomax said. Other surveys collectively show a rapid slowdown in UK house price growth, she pointed out. This means that despite a strong economic growth, base rates will probably stay on hold at 4.75%. Official data comes from the Office for National Statistics (ONS). Though reliable, ONS data takes longer to publish, so now the BoE is calling for faster delivery of data so it can make more effective policy decisions. "Recent work by the Bank has shown that private sector surveys add value, even when preliminary ONS estimates are available," Ms Lomax said in a speech to the North Wales Business Club.\n\nThe ONS is due to publish its second estimate of third quarter growth on Friday. "The MPC judges that

In [34]:
print(train_dataset.features)

{'Text': Value('string'), 'Summary': Value('string'), 'Category': Value('string')}


In [35]:
print(f"Train Samples      : {len(train_dataset)}")
print(f"Validation Samples : {len(val_dataset)}")
print(f"Test Samples       : {len(test_dataset)}")

Train Samples      : 400
Validation Samples : 50
Test Samples       : 50


In [36]:
from transformers import BartTokenizer

In [85]:
model_name = "sshleifer/distilbart-cnn-12-6"

tokenizer = BartTokenizer.from_pretrained(model_name)

print("Tokenizer Loaded Successfully!")

Tokenizer Loaded Successfully!


In [38]:
print("ARTICLE\n")
print(train_dataset[0]["Text"][:500])

print("\n")

print("SUMMARY\n")
print(train_dataset[0]["Summary"])

ARTICLE

Economy 'stronger than forecast'

The UK economy probably grew at a faster rate in the third quarter than the 0.4% reported, according to Bank of England deputy governor Rachel Lomax.

Private sector business surveys suggest a stronger economy than official estimates, Ms Lomax said. Other surveys collectively show a rapid slowdown in UK house price growth, she pointed out. This means that despite a strong economic growth, base rates will probably stay on hold at 4.75%. Official data comes from t


SUMMARY

"The MPC judges that overall growth was a little higher in the third quarter than the official data currently indicate," Ms Lomax said."Recent work by the Bank has shown that private sector surveys add value, even when preliminary ONS estimates are available," Ms Lomax said in a speech to the North Wales Business Club.Private sector business surveys suggest a stronger economy than official estimates, Ms Lomax said."This year rising oil prices and a significant slowdown in th

In [86]:
sample = tokenizer(
    train_dataset[0]["Text"],
    max_length=512,
    truncation=True,
    padding="max_length"
)

print(sample.keys())

KeysView({'input_ids': [0, 44494, 219, 128, 8355, 254, 87, 1914, 108, 50118, 50118, 133, 987, 866, 1153, 2307, 23, 10, 3845, 731, 11, 5, 371, 297, 87, 5, 321, 4, 306, 207, 431, 6, 309, 7, 788, 9, 1156, 3193, 2318, 7423, 226, 1075, 3631, 4, 50118, 50118, 38076, 1293, 265, 12092, 3608, 10, 3651, 866, 87, 781, 2785, 6, 2135, 226, 1075, 3631, 26, 4, 1944, 12092, 14332, 311, 10, 6379, 9831, 11, 987, 790, 425, 434, 6, 79, 3273, 66, 4, 152, 839, 14, 1135, 10, 670, 776, 434, 6, 1542, 1162, 40, 1153, 1095, 15, 946, 23, 204, 4, 2545, 2153, 13485, 414, 606, 31, 5, 1387, 13, 496, 12064, 36, 28663, 322, 3791, 7058, 6, 5121, 104, 414, 1239, 1181, 7, 10732, 6, 98, 122, 5, 3542, 717, 16, 1765, 13, 3845, 2996, 9, 414, 98, 24, 64, 146, 55, 2375, 714, 2390, 4, 22, 39936, 173, 30, 5, 788, 34, 2343, 14, 940, 1293, 12092, 1606, 923, 6, 190, 77, 6104, 5121, 104, 2785, 32, 577, 60, 2135, 226, 1075, 3631, 26, 11, 10, 1901, 7, 5, 369, 5295, 2090, 2009, 4, 50118, 50118, 133, 5121, 104, 16, 528, 7, 10732, 63, 200

In [87]:
max_input_length = 512
max_target_length = 128

In [89]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [90]:
tokenized_train = tokenized_train.remove_columns(
    ["Text", "Summary", "Category"]
)

tokenized_val = tokenized_val.remove_columns(
    ["Text", "Summary", "Category"]
)

tokenized_test = tokenized_test.remove_columns(
    ["Text", "Summary", "Category"]
)

In [46]:
print(tokenized_train.features)

{'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}


In [47]:
print(tokenized_train[0])

{'input_ids': [0, 44494, 219, 128, 8355, 254, 87, 1914, 108, 50118, 50118, 133, 987, 866, 1153, 2307, 23, 10, 3845, 731, 11, 5, 371, 297, 87, 5, 321, 4, 306, 207, 431, 6, 309, 7, 788, 9, 1156, 3193, 2318, 7423, 226, 1075, 3631, 4, 50118, 50118, 38076, 1293, 265, 12092, 3608, 10, 3651, 866, 87, 781, 2785, 6, 2135, 226, 1075, 3631, 26, 4, 1944, 12092, 14332, 311, 10, 6379, 9831, 11, 987, 790, 425, 434, 6, 79, 3273, 66, 4, 152, 839, 14, 1135, 10, 670, 776, 434, 6, 1542, 1162, 40, 1153, 1095, 15, 946, 23, 204, 4, 2545, 2153, 13485, 414, 606, 31, 5, 1387, 13, 496, 12064, 36, 28663, 322, 3791, 7058, 6, 5121, 104, 414, 1239, 1181, 7, 10732, 6, 98, 122, 5, 3542, 717, 16, 1765, 13, 3845, 2996, 9, 414, 98, 24, 64, 146, 55, 2375, 714, 2390, 4, 22, 39936, 173, 30, 5, 788, 34, 2343, 14, 940, 1293, 12092, 1606, 923, 6, 190, 77, 6104, 5121, 104, 2785, 32, 577, 60, 2135, 226, 1075, 3631, 26, 11, 10, 1901, 7, 5, 369, 5295, 2090, 2009, 4, 50118, 50118, 133, 5121, 104, 16, 528, 7, 10732, 63, 200, 3278, 9

In [91]:
decoded_article = tokenizer.decode(
    tokenized_train[0]["input_ids"],
    skip_special_tokens=True
)

decoded_summary = tokenizer.decode(
    tokenized_train[0]["labels"],
    skip_special_tokens=True
)

print("=" * 60)
print("Decoded Article")
print("=" * 60)
print(decoded_article[:500])

print("\n")

print("=" * 60)
print("Decoded Summary")
print("=" * 60)
print(decoded_summary)

Decoded Article
Economy 'stronger than forecast'

The UK economy probably grew at a faster rate in the third quarter than the 0.4% reported, according to Bank of England deputy governor Rachel Lomax.

Private sector business surveys suggest a stronger economy than official estimates, Ms Lomax said. Other surveys collectively show a rapid slowdown in UK house price growth, she pointed out. This means that despite a strong economic growth, base rates will probably stay on hold at 4.75%. Official data comes from t


Decoded Summary
"The MPC judges that overall growth was a little higher in the third quarter than the official data currently indicate," Ms Lomax said."Recent work by the Bank has shown that private sector surveys add value, even when preliminary ONS estimates are available," Ms Lomax said in a speech to the North Wales Business Club.Private sector business surveys suggest a stronger economy than official estimates, Ms Lomax said."This year rising oil prices and a significant 

In [49]:
print("Training :", len(tokenized_train))

print("Validation :", len(tokenized_val))

print("Test :", len(tokenized_test))

Training : 400
Validation : 50
Test : 50


In [50]:
print(tokenized_train.features)

{'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}


In [92]:
from transformers import (
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)

In [93]:
from transformers import BartForConditionalGeneration

model = BartForConditionalGeneration.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model running on:", device)

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model running on: cuda


In [94]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print("Data Collator Created")

Data Collator Created


In [108]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./distilbart_bbc",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    learning_rate=2e-5,
    remove_unused_columns=False,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=2,

    warmup_steps=50,

    weight_decay=0.01,

    lr_scheduler_type="linear",

    optim="adamw_torch",

    predict_with_generate=True,

    logging_dir="./logs",

    report_to="tensorboard",

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    fp16=torch.cuda.is_available(),

    seed=42
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [109]:
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=1
)

In [68]:
print("Learning Rate :", training_args.learning_rate)
print("Batch Size :", training_args.per_device_train_batch_size)
print("Epochs :", training_args.num_train_epochs)
print("Warmup Steps :", training_args.warmup_steps)
print("Optimizer :", training_args.optim)
print("Scheduler :", training_args.lr_scheduler_type)

Learning Rate : 2e-05
Batch Size : 4
Epochs : 2
Warmup Steps : 50
Optimizer : OptimizerNames.ADAMW_TORCH
Scheduler : SchedulerType.LINEAR


In [97]:
from transformers import Seq2SeqTrainer
import inspect

print(inspect.signature(Seq2SeqTrainer))

(model: Union[ForwardRef('PreTrainedModel'), torch.nn.modules.module.Module, NoneType] = None, args: Optional[ForwardRef('TrainingArguments')] = None, data_collator: Optional[ForwardRef('DataCollator')] = None, train_dataset: Union[torch.utils.data.dataset.Dataset, ForwardRef('IterableDataset'), ForwardRef('datasets.Dataset'), NoneType] = None, eval_dataset: torch.utils.data.dataset.Dataset | dict[str, torch.utils.data.dataset.Dataset] | None = None, processing_class: Union[ForwardRef('PreTrainedTokenizerBase'), ForwardRef('BaseImageProcessor'), ForwardRef('FeatureExtractionMixin'), ForwardRef('ProcessorMixin'), NoneType] = None, model_init: collections.abc.Callable[[], 'PreTrainedModel'] | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[['EvalPrediction'], dict] | None = None, callbacks: list['TrainerCallback'] | None = None, optimizers: tuple[torch.optim.optimizer.Optimizer | None, torch.optim.lr_scheduler.LambdaLR | N

In [110]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    callbacks=[early_stopping]
)

In [111]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.244786,1.453745
2,0.986316,0.804687


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=100, training_loss=1.6155509567260742, metrics={'train_runtime': 155.1038, 'train_samples_per_second': 5.158, 'train_steps_per_second': 0.645, 'total_flos': 619164834201600.0, 'train_loss': 1.6155509567260742, 'epoch': 2.0})

In [101]:
def preprocess_function(examples):

    model_inputs = tokenizer(
        examples["Text"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["Summary"],   # <- IMPORTANT
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [102]:
train_dataset = train_dataset.map(preprocess_function, batched=True)

val_dataset = val_dataset.map(preprocess_function, batched=True)

test_dataset = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [106]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [112]:
save_path = "./distilbart_bbc_finetuned"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [113]:
from transformers import BartTokenizer, BartForConditionalGeneration

loaded_tokenizer = BartTokenizer.from_pretrained(save_path)

loaded_model = BartForConditionalGeneration.from_pretrained(save_path)

print("Model reloaded successfully!")

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model reloaded successfully!


In [114]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaded_model.to(device)

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [116]:
new_article = """
Artificial Intelligence is increasingly being used in healthcare to assist doctors
in diagnosing diseases, analyzing medical images, predicting patient outcomes,
and improving treatment planning. Hospitals around the world are adopting AI-based
systems to increase efficiency and provide better patient care.
"""

inputs = loaded_tokenizer(
    new_article,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)

summary_ids = loaded_model.generate(
    inputs["input_ids"],
    max_length=100,
    min_length=10,
    num_beams=4
)

summary = loaded_tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Generated Summary:\n")
print(summary)

Generated Summary:

Artificial Intelligence is increasingly being used in healthcare to assist doctors in diagnosing diseases, analyzing medical images, predicting patient outcomes,and improving treatment planning.Artificial Artificial Intelligence is also being used to help doctors and nurses diagnose diseases, analyze medical images and predict patient outcomes. Hospitals around the world are adopting AI-based systems to increase efficiency and provide better patient care.
